In [ ]:
# # 补充实验：EP 赛马、模型互相定价与 CH-4 检验
#
# 先读 `src_jup/CH3_核心解释实验.ipynb` 的研究主线结论。这个 notebook 按补充问题组织实现：先用 Fama–MacBeth 比较 EP、BM、AM、CP；再比较 CH-3 与 FF-3 的因子定价误差并做共同样本 Gibbons–Ross–Shanken（GRS）检验；随后报告换手因子加入后的 CH-4 定价结果。剩余章节保留论文逐表复现、A1/A4/A5 和覆盖矩阵，便于逐项核对原文。
#
# 输入是 `01_data_and_factors.ipynb` 生成的同一组 S5 面板和因子。每项输出区分论文原值、本地计算值和口径限制；短样本不补成原文月份，代理变量不标成原变量。
#
from IPython.display import HTML
import matplotlib
matplotlib.use('Agg')
from IPython.core.pylabtools import print_figure
from matplotlib.figure import Figure
get_ipython().display_formatter.formatters['image/png'].for_type(Figure, lambda figure:print_figure(figure,fmt='png'))

computed_rows = []


def record_result(table,panel,row,metric,value,months,status='本地数据计算，口径见限制表'):
    computed_rows.append({'table':table,'panel':panel,'row':row,'metric':metric,'computed_value':value,'months':months,'status':status})


def run_ts(y, f):
    y = y.rename('dependent_return')
    aligned = pd.concat([y,f],axis=1,join='inner').replace([np.inf,-np.inf],np.nan).dropna()
    if len(aligned) <= f.shape[1]+2:
        return None
    x = sm.add_constant(aligned[f.columns],has_constant='add')
    if np.linalg.matrix_rank(x) < x.shape[1]:
        return None
    return sm.OLS(aligned['dependent_return'],x).fit(cov_type='HC1')


def grs_test(r, f):
    aligned = pd.concat([r,f],axis=1,join='inner').replace([np.inf,-np.inf],np.nan).dropna()
    t = len(aligned)
    n = r.shape[1]
    k = f.shape[1]
    if n == 0 or t <= n+k:
        return {'grs':np.nan,'p':np.nan,'months':t,'assets':n}
    y = aligned[r.columns].to_numpy()
    factors = aligned[f.columns].to_numpy()
    x = np.column_stack([np.ones(t),factors])
    if np.linalg.matrix_rank(x)<k+1:
        return {'grs':np.nan,'p':np.nan,'months':t,'assets':n}
    coefficient = np.linalg.lstsq(x,y,rcond=None)[0]
    residual = y-x@coefficient
    sigma = residual.T@residual/t
    if np.linalg.matrix_rank(sigma)<n:
        return {'grs':np.nan,'p':np.nan,'months':t,'assets':n}
    alpha = coefficient[0]
    denominator = 1.0
    if k:
        mean = factors.mean(axis=0)
        centered = factors-mean
        omega = centered.T@centered/t
        if np.linalg.matrix_rank(omega)<k:
            return {'grs':np.nan,'p':np.nan,'months':t,'assets':n}
        denominator += mean@np.linalg.solve(omega,mean)
    statistic = (t-n-k)/n * (alpha@np.linalg.solve(sigma,alpha)) / denominator
    return {'grs':float(statistic),'p':float(stats.f.sf(statistic,n,t-n-k)),'months':t,'assets':n}


def get_mean_stats(y):
    y = y.replace([np.inf,-np.inf],np.nan).dropna()
    if len(y)<3:
        return np.nan,np.nan,0
    result = sm.OLS(y.to_numpy(),np.ones((len(y),1))).fit(cov_type='HC1')
    return float(result.params[0]),float(result.tvalues[0]),len(y)

print('全市场每月交易日数低于15：',df.groupby(form)[date].nunique().loc[lambda x:x<15].to_dict())
print('股票池缺少的收益月：',pd.period_range('2000-01','2016-12',freq='M').difference(f_ch3.index).astype(str).tolist())


In [ ]:
# 回归准备：按论文 Table 2 口径计算前置 CAPM beta.
# 使用形成月前12个月的日收益，包含当日与五个滞后市场收益；估计后合并到回归样本。
# 本单元依赖 CH3_jup/reproduction_pipeline/01_data_and_factors.ipynb 先生成 df、snap 与 monthly_final。

previous_cap = df.groupby(stock_code)[day_market_value].shift(1)
market_parts = pd.DataFrame(
    {
        date: df[date],
        'weight': previous_cap,
        'contribution': previous_cap * df[daily_ret],
    }
)
market_parts = market_parts[market_parts['weight'] > 0]
market_parts = market_parts[market_parts['contribution'].notna()]
daily_market = market_parts.groupby(date)[
    ['weight', 'contribution']
].sum()
daily_market['market_return'] = (
    daily_market['contribution'] / daily_market['weight']
)
market_lags = pd.concat(
    [
        daily_market['market_return']
        .shift(lag)
        .rename('market_lag_{}'.format(lag))
        for lag in range(6)
    ],
    axis=1,
)
del market_parts
del previous_cap


def get_dimson_beta(df, snap, market_lags):
    data_list = []
    signal_groups = dict(list(snap.groupby(stock_code, sort=False)))
    for code_value, data in df.groupby(stock_code, sort=False):
        signal = signal_groups.get(code_value, pd.DataFrame())
        if signal.empty:
            continue
        x = market_lags.reindex(data[date]).to_numpy()
        y = data[daily_ret].to_numpy()
        valid = np.isfinite(x).all(axis=1) & np.isfinite(y)
        x = np.column_stack([np.ones(valid.sum()),x[valid]])
        y = y[valid]
        dates = data.loc[valid,date].to_numpy()
        cross_products = x[:, :, None] * x[:, None, :]
        cross_products = np.cumsum(cross_products, axis=0)
        cross_products = np.concatenate(
            [np.zeros((1, 7, 7)), cross_products],
            axis=0,
        )
        return_products = x * y[:, None]
        return_products = np.cumsum(return_products, axis=0)
        return_products = np.vstack([np.zeros((1, 7)), return_products])
        end = signal['formation_end']
        begin = end - pd.DateOffset(months=12)
        left = np.searchsorted(
            dates, begin.to_numpy(), side='right'
        )
        right = np.searchsorted(dates, end.to_numpy(), side='right')
        betas = np.repeat(np.nan, len(signal))
        for position in range(len(signal)):
            # 论文未另设最少日观测门槛；此处仅要求七个系数可识别且保留正残差自由度。
            if right[position] - left[position] <= 7:
                continue
            xx = (
                cross_products[right[position]]
                - cross_products[left[position]]
            )
            xy = (
                return_products[right[position]]
                - return_products[left[position]]
            )
            if np.linalg.matrix_rank(xx) < 7:
                continue
            coefficient = np.linalg.solve(xx, xy)
            betas[position] = coefficient[1:].sum()
        beta_table = pd.DataFrame(
            {
                stock_code: code_value,
                form: signal[form].to_numpy(),
                'monthly_beta': betas,
            }
        )
        data_list.append(beta_table)
    return pd.concat(data_list, ignore_index=True)


beta = get_dimson_beta(df, snap, market_lags)
monthly_final = monthly_final.merge(
    beta,
    on=[stock_code, form],
    how='left',
    validate='one_to_one',
)
print('Table 2 beta sample:', monthly_final.shape, monthly_final[ym].nunique())
beta_columns = [
    monthly_market_value,
    monthly_am,
    monthly_ep,
    monthly_bm,
    monthly_cp,
    monthly_asset_growth,
    monthly_accrual,
    monthly_noa,
    'monthly_beta',
]
beta_coverage = monthly_final.groupby(ym)[beta_columns].count()
display(beta_coverage.iloc[[0, -1]])


In [ ]:
# ## 补充实验一：估值比率赛马（论文 Table 2）→ 月度截面回归 → NW(4)
#
# `log_me` 用百万元市值，改变金额单位只改变截距，不改变规模斜率；原文没有明确给出 logME 的金额基准，截距可比性单独记录。`log_bm/log_am` 要求正值；EP+/CP+ 保留 0，负值 dummy 用 `<0`，不是 `<=0`。
#
# 估值比率赛马集中比较规格(4) log BM、(5) log AM、(6) EP+/负EP dummy、(7) CP+/负CP dummy、(8) 全部估值变量共同进入、(9) BM 与 EP 共同进入。各列都控制滞后 beta 与 log 市值；对比 EP+ 的平均斜率、NW(4) t 值及加入其他比率前后的变化，判断 EP 是否吸收其他价值信号。论文以此选 EP 构造价值因子。当前数据的 EP 是 EPS/收盘价代理，结果作为本地赛马，不能冒充扣非盈利的原口径。
#
# 输入每行股票×持有月；按每个规格自己的共同有效股票做 OLS。输出 `slopes` 一行持有月、列为截距/特征系数。NW 是系数序列上的四阶 HAC；不对本月横截面套 HAC，不用恒定的暴露矩阵 B 代替滞后特征。
#
# 四阶 NW 按自然月距离计算：先把斜率残差恢复完整月历，缺月保留NaN，滞后乘积只使用两个端点都有效的月份。不能把2月与4月压缩为相邻观测。连续月历下该公式与statsmodels HAC(4)一致。
#
def get_calendar_nw_mean(sequence, lags=4):
    valid = sequence.replace([np.inf,-np.inf],np.nan).dropna()
    count = len(valid)
    if count<3:
        return np.nan,np.nan,count
    mean = valid.mean()
    calendar = pd.period_range(valid.index.min(),valid.index.max(),freq='M')
    residual = (valid-mean).reindex(calendar)
    variance_sum = residual.pow(2).sum()
    for lag in range(1,lags+1):
        variance_sum += 2*(1-lag/(lags+1))*(residual*residual.shift(lag)).sum()
    se = np.sqrt(max(variance_sum,0))/count
    return float(mean),float(mean/se) if se>0 else np.nan,count


fm_specs = {
    '(1)':['beta'], '(2)':['log_me'], '(3)':['beta','log_me'],
    '(4)':['beta','log_me','log_bm'], '(5)':['beta','log_me','log_am'],
    '(6)':['beta','log_me','ep_pos','ep_neg'], '(7)':['beta','log_me','cp_pos','cp_neg'],
    '(8)':['beta','log_me','log_bm','log_am','ep_pos','ep_neg','cp_pos','cp_neg'],
    '(9)':['beta','log_me','log_bm','ep_pos','ep_neg']
}


def get_characteristic_panel(monthly_final):
    data = monthly_final.copy()
    data['beta'] = data['monthly_beta']
    data['log_me'] = np.log(data[monthly_market_value].where(data[monthly_market_value].gt(0))/1000000)
    data['log_bm'] = np.log(data[monthly_bm].where(data[monthly_bm].gt(0)))
    data['log_am'] = np.log(data[monthly_am].where(data[monthly_am].gt(0)))
    for prefix,column in [('ep',monthly_ep),('cp',monthly_cp)]:
        data[prefix+'_pos'] = data[column].clip(lower=0)
        data[prefix+'_neg'] = data[column].lt(0).astype(float).where(data[column].notna())
    return data


def get_characteristic_fm(data):
    result_rows = []
    slope_tables = {}
    for specification,features in fm_specs.items():
        slopes_list = []
        for month,group in data.groupby(ym):
            valid = group[[monthly_return]+features].replace([np.inf,-np.inf],np.nan).dropna()
            if len(valid)<=len(features)+2:
                continue
            x = sm.add_constant(valid[features],has_constant='add')
            if np.linalg.matrix_rank(x)<x.shape[1]:
                continue
            result = sm.OLS(valid[monthly_return],x).fit()
            row = result.params.rename(index={'const':'intercept'}).to_dict()
            row[ym] = month
            row['adj_r2'] = result.rsquared_adj
            row['stock_count'] = len(valid)
            slopes_list.append(row)
        if not slopes_list:
            slope_tables[specification] = pd.DataFrame()
            continue
        slopes = pd.DataFrame(slopes_list).set_index(ym).sort_index()
        slope_tables[specification] = slopes
        for feature in ['intercept']+features:
            sequence = slopes[feature].dropna()
            estimate,t_stat,count = get_calendar_nw_mean(sequence,4)
            result_rows.append({'panel':specification,'row':feature,'estimate':estimate,'t':t_stat,'months':count,'mean_stocks':slopes['stock_count'].mean()})
        result_rows.append({'panel':specification,'row':'adj_r2','estimate':slopes['adj_r2'].mean(),'t':np.nan,'months':len(slopes),'mean_stocks':slopes['stock_count'].mean()})
    return pd.DataFrame(result_rows),slope_tables


characteristic_panel = get_characteristic_panel(monthly_final)
table2_result,fm_slopes = get_characteristic_fm(characteristic_panel)
for row in table2_result.to_dict('records'):
    for metric in ['estimate','t']:
        record_result('table2',row['panel'],row['row'],metric,row[metric],row['months'])
display(table2_result)
print('规格(6)的规模斜率与 EP+ 斜率相关系数：',fm_slopes['(6)'][['log_me','ep_pos']].corr().iloc[0,1])


In [ ]:
# ## 表 3：CH-3 溢价、波动与相关
#
# 均值和标准差用月度百分数，Table 3 的均值 t 用 `mean/(std/√T)`；其他 alpha 表用 White HC1。简单年化溢价是月均值×12，与财富累计复利分开。
#
table3_result = pd.DataFrame({'mean':100*f_ch3.mean(),'std':100*f_ch3.std(ddof=1),'t_mean':f_ch3.mean()/(f_ch3.std(ddof=1)/np.sqrt(f_ch3.count())),'months':f_ch3.count()})
correlation = f_ch3.corr()
for column,row_name in [(mkt,'mkt'),(smb,'smb'),(vmg,'vmg')]:
    for metric in ['mean','std','t_mean']:
        record_result('table3','main',row_name,metric,table3_result.loc[column,metric],f_ch3[column].count())
    for other,other_name in [(mkt,'mkt'),(smb,'smb'),(vmg,'vmg')]:
        record_result('table3','main',row_name,'corr_'+other_name,correlation.loc[column,other],f_ch3[[column,other]].dropna().shape[0])
display(table3_result)
display(correlation)
print('SMB、VMG 简单年化均值（%）：',(100*12*f_ch3[[smb,vmg]].mean()).to_dict())
for column in f_ch3:
    result = run_ts(f_ch3[column],f_ch3.drop(columns=column))
    print('两因子对 {}：alpha {:.4f}% / t {:.3f}'.format(column,100*result.params['const'],result.tvalues['const']))
figure,axis = plt.subplots(figsize=(8,4))
actual = 100*f_ch3.mean().rename(index={mkt:'MKT',smb:'SMB',vmg:'VMG'})
pd.DataFrame({'论文原值':[0.66,1.03,1.14],'RESSET代理计算':actual.to_numpy()},index=['MKT','SMB','VMG']).plot.bar(ax=axis,color=['#adb5bd','#2f6690'])
axis.set_ylabel('月均收益（%）')
axis.set_title('表3：原值与本地代理口径并排')
axis.tick_params(axis='x',rotation=0)
figure.tight_layout()
display(figure)
plt.close(figure)


In [ ]:
# ## 表 4：36 月滚动解释度，先时间平均再股票平均
#
# Panel A 用月表中所有 A 股有效收益，包含小股票；Panel B 用窗口结束月属于 top70 股票池的股票，但回归窗口仍取该股票此前 36 个自然月收益。每个窗口要求完整 36 月，这一最低样本规则为本次明确实施口径。各股票先在窗口间平均，最后对股票平均，不把所有股票窗口混成同一个均值。
#
# 美国 Panel C 缺少 CRSP 与美国 FF 因子，保留原值和缺数状态。
#
def get_rolling_r2(monthly, monthly_final, factors):
    factors = factors.reindex(pd.period_range('2000-01','2016-12',freq='M'))
    returns = monthly.pivot(index=ym,columns=stock_code,values=monthly_excess_ret).reindex(factors.index)
    membership = monthly_final.assign(member=True).pivot(index=ym,columns=stock_code,values='member').reindex(index=returns.index,columns=returns.columns).fillna(False)
    models = {'mkt':[mkt],'mkt_smb':[mkt,smb],'mkt_value':[mkt,vmg],'three_factor':[mkt,smb,vmg]}
    results = []
    for model,columns in models.items():
        x = np.column_stack([np.ones(len(factors)),factors[columns].to_numpy()])
        valid_factor = np.isfinite(x).all(axis=1)
        x = np.nan_to_num(x)
        prefix_xx = np.concatenate([np.zeros((1,x.shape[1],x.shape[1])),np.cumsum(x[:,:,None]*x[:,None,:],axis=0)])
        xx = prefix_xx[36:]-prefix_xx[:-36]
        factor_complete = pd.Series(valid_factor).rolling(36).sum().to_numpy()[35:]==36
        all_averages = []
        top_averages = []
        windows_all = 0
        windows_top = 0
        for column in returns:
            y = returns[column].to_numpy()
            valid_y = np.isfinite(y)
            filled = np.nan_to_num(y)
            prefix_xy = np.vstack([np.zeros((1,x.shape[1])),np.cumsum(x*filled[:,None],axis=0)])
            xy = prefix_xy[36:]-prefix_xy[:-36]
            prefix_yy = np.r_[0,np.cumsum(filled**2)]
            yy = prefix_yy[36:]-prefix_yy[:-36]
            sst = yy-xy[:,0]**2/36
            complete = (pd.Series(valid_y).rolling(36).sum().to_numpy()[35:]==36) & factor_complete & (sst>1e-15)
            positions = np.flatnonzero(complete)
            if len(positions)==0:
                continue
            coefficient = np.linalg.solve(xx[positions],xy[positions,:,None])[:,:,0]
            r2 = 1-(yy[positions]-(coefficient*xy[positions]).sum(axis=1))/sst[positions]
            all_averages.append(r2.mean())
            windows_all += len(r2)
            top = membership[column].to_numpy()[35:][positions].astype(bool)
            if top.any():
                top_averages.append(r2[top].mean())
                windows_top += int(top.sum())
        for panel,values,count in [('china_all',all_averages,windows_all),('china_top70',top_averages,windows_top)]:
            results.append({'panel':panel,'row':model,'r2':np.mean(values) if values else np.nan,'stocks':len(values),'windows':count})
    return pd.DataFrame(results)


table4_result = get_rolling_r2(monthly,monthly_final,f_ch3)
for row in table4_result.to_dict('records'):
    record_result('table4',row['panel'],row['row'],'r2',row['r2'],int(f_ch3.dropna().shape[0]))
display(table4_result)


In [ ]:
# ## 补充实验二：CH-3 与 FF-3 相互定价及 GRS（论文 Table 5 / A4 / A5）
#
# 时序回归只用 y 与 F 的共同完整月份；White HC1 影响 t 值，经典 GRS 则使用同一组完整观测的 OLS 残差协方差。GRS 不能拿各资产不同样本估计的 alpha 拼接，也不能将 White 方差直接塞入经典 F 检验。
#
# FF-5 的盈利使用已披露 ROE 代理 operating profitability，投资因子因财务缺数只能覆盖后半段。这部分为明确标记的敏感性比较，不能声称精确重现 Table A5。
#
factor_ids = {smb:'smb',vmg:'vmg','ff_size_factor':'ffsmb','ff_value_factor':'ffhml','ff_profit_factor':'ffrmw','ff_invest_factor':'ffcma'}
metric_ids = {mkt:'mkt',smb:'smb',vmg:'vmg','ff_size_factor':'ffsmb','ff_value_factor':'ffhml','ff_profit_factor':'ffrmw','ff_invest_factor':'ffcma'}
model_comparison = []
for source,target,target_name,table in [(f_ff3,f_ch3,'ch3','table5'),(f_ch3,f_ff3,'ff3','table5'),(f_ff5,f_ch3,'ch3','a5'),(f_ch3,f_ff5,'ff5','a5')]:
    for column in source.columns.drop(mkt):
        result = run_ts(source[column],target)
        if result is None:
            continue
        row_name = factor_ids[column]
        record_result(table,'alpha',row_name,'alpha_'+target_name,100*result.params['const'],int(result.nobs))
        record_result(table,'alpha',row_name,'t_alpha_'+target_name,result.tvalues['const'],int(result.nobs))
        model_comparison.append({'table':table,'factor':row_name,'model':target_name,'alpha':100*result.params['const'],'t_alpha':result.tvalues['const'],'months':int(result.nobs)})
        if table=='table5':
            mean,t_mean,count = get_mean_stats(source[column])
            for metric,value in [('mean',100*mean),('t_mean',t_mean),('alpha',100*result.params['const']),('t_alpha',result.tvalues['const'])]:
                record_result('a4','main',row_name,metric,value,int(result.nobs))
            for regressor in target:
                record_result('a4','main',row_name,'beta_'+metric_ids[regressor],result.params[regressor],int(result.nobs))
                record_result('a4','main',row_name,'t_beta_'+metric_ids[regressor],result.tvalues[regressor],int(result.nobs))
    test = grs_test(source.drop(columns=mkt),target)
    group_name = '_'.join(factor_ids[column] for column in source.columns.drop(mkt))
    for metric in ['grs','p']:
        record_result(table,'grs',group_name,metric+'_'+target_name,test[metric],test['months'])
display(pd.DataFrame(model_comparison))


In [ ]:
# ## 补充实验三：异象多空组合与 CH-4 定价能力（论文 Table 6 / 7 / 8 / 10 / A1）
#
# 正文十个异象固定为原文的十行，附录 A1 报告全部十四个。保留未通过本地 CAPM 的正文行，不能为了凑足“十个显著”重新筛掉或加入本地异象。Panel B 省略规模，实际九行。
#
# 所有 alpha 与均值显示为月百分比，beta 不乘 100；结果同时给出 5% 和 10% 双侧判断。原文 Table 10 的 size-neutral MAX t=-2.05 与“全部解释”的概括存在边界矛盾，单独报告。
#
main_anomalies = ['size','ep','bm','cp','roe','volatility','max','reversal','turnover','abnormal_turnover']
capm = f_ch3[[mkt]]
anomaly_regressions = {}
all_model_rows = []
for panel,returns in [('unconditional',anomaly_uncon),('size_neutral',anomaly_neutral)]:
    for model,factors,table in [('capm',capm,'table6'),('ch3',f_ch3,'table7'),('ff3',f_ff3,'table8'),('ch4',f_ch4,'table10')]:
        rows = []
        for name in returns.columns:
            result = run_ts(returns[name],factors)
            if result is None:
                continue
            mean,t_mean,count = get_mean_stats(returns[name])
            row = {'row':name,'mean':100*mean,'t_mean':t_mean,'alpha':100*result.params['const'],'t_alpha':result.tvalues['const'],'months':int(result.nobs)}
            for column in factors:
                metric_name = {mkt:'mkt',smb:'smb',vmg:'vmg','ff_size_factor':'smb','ff_value_factor':'hml','turnover_factor_pmo':'pmo'}[column]
                row['beta_'+metric_name] = result.params[column]
                row['t_beta_'+metric_name] = result.tvalues[column]
            row['significant_5pct'] = abs(row['t_alpha'])>=stats.norm.ppf(0.975)
            row['significant_10pct'] = abs(row['t_alpha'])>=stats.norm.ppf(0.95)
            rows.append(row)
            if model=='capm':
                for metric in ['alpha','t_alpha']:
                    record_result('a1',panel,name,metric,row[metric],row['months'])
            if name in main_anomalies:
                for metric,value in row.items():
                    if metric not in ['row','months','significant_5pct','significant_10pct']:
                        record_result(table,panel,name,metric,value,row['months'])
                all_model_rows.append({'panel':panel,'model':model,**row})
        anomaly_regressions[(panel,model)] = pd.DataFrame(rows).set_index('row')
        print(table, panel)
        display(anomaly_regressions[(panel,model)].reindex([name for name in main_anomalies if name in returns.columns]))


In [ ]:
# ## 表 9：同一资产集合上的模型比较
#
# 分别报告原始均值、CAPM、FF-3、CH-3 的平均 |alpha|、平均 |t|、全资产 GRS 与七资产 GRS；Panel A 十行、Panel B 九行，七资产版都剔除 size/EP/BM。
#
# 另输出共同样本的比较，避免 CP 缺数造成某模型在更多月份估计。GRS 的 n 是资产数量、k 是解释因子数量、t 是共同完整月份数。
#
table9_rows = []
for panel,returns in [('unconditional',anomaly_uncon),('size_neutral',anomaly_neutral)]:
    names = [name for name in main_anomalies if name in returns]
    seven = [name for name in names if name not in ['size','ep','bm']]
    for model,factors in [('unadjusted',pd.DataFrame(index=f_ch3.index)),('capm',capm),('ff3',f_ff3),('ch3',f_ch3)]:
        alpha_values = []
        t_values = []
        sample_counts = []
        for name in names:
            result = run_ts(returns[name],factors)
            if result is not None:
                alpha_values.append(100*result.params['const'])
                t_values.append(result.tvalues['const'])
                sample_counts.append(int(result.nobs))
        full = grs_test(returns[names],factors)
        reduced = grs_test(returns[seven],factors)
        row = {'panel':panel,'row':model,'avg_abs_alpha':np.mean(np.abs(alpha_values)),'avg_abs_t':np.mean(np.abs(t_values)),'grs_all':full['grs'],'p_all':full['p'],'grs7':reduced['grs'],'p7':reduced['p'],'months_all':full['months'],'assets_all':full['assets'],'min_individual_months':min(sample_counts),'max_individual_months':max(sample_counts)}
        table9_rows.append(row)
        for metric in ['avg_abs_alpha','avg_abs_t','grs_all','p_all','grs7','p7']:
            months = (sample_counts[0] if len(set(sample_counts))==1 else np.nan) if metric in ['avg_abs_alpha','avg_abs_t'] else (reduced['months'] if metric in ['grs7','p7'] else full['months'])
            status = '单项alpha样本范围{}—{}月；GRS用共同完整样本'.format(min(sample_counts),max(sample_counts))
            record_result('table9',panel,model,metric,row[metric],months,status=status)
table9_result = pd.DataFrame(table9_rows)
display(table9_result)
print('CH-4：平均绝对 alpha、t 与 GRS')
for panel,returns in [('unconditional',anomaly_uncon),('size_neutral',anomaly_neutral)]:
    names = [name for name in main_anomalies if name in returns]
    estimates = anomaly_regressions[(panel,'ch4')].reindex(names)
    print(panel, 'mean_abs_alpha',estimates['alpha'].abs().mean(),'mean_abs_t',estimates['t_alpha'].abs().mean(),'grs',grs_test(returns[names],f_ch4))


common_summary_rows = []
for panel,returns in [('unconditional',anomaly_uncon),('size_neutral',anomaly_neutral)]:
    names = [name for name in main_anomalies if name in returns]
    common_index = returns[names].dropna().index.intersection(f_ch3.dropna().index).intersection(f_ff3.dropna().index).intersection(f_ch4.dropna().index)
    for model,factors in [('unadjusted',pd.DataFrame(index=common_index)),('capm',capm),('ff3',f_ff3),('ch3',f_ch3),('ch4',f_ch4)]:
        model_index = common_index.intersection(factors.dropna().index) if len(factors.columns) else common_index
        common_returns = returns.loc[model_index,names]
        common_factors = factors.reindex(model_index)
        estimates = [run_ts(common_returns[name],common_factors) for name in names]
        joint = grs_test(common_returns,common_factors)
        common_summary_rows.append({'panel':panel,'model':model,'avg_abs_alpha':np.mean([100*abs(result.params['const']) for result in estimates]),'avg_abs_t':np.mean([abs(result.tvalues['const']) for result in estimates]),'grs':joint['grs'],'p':joint['p'],'months':len(model_index),'assets':len(names)})
table9_common_result = pd.DataFrame(common_summary_rows)
print('每个模型、每项异象均使用共同完整月份：')
display(table9_common_result)


In [ ]:
# ## 逐格对照：先看数据身份，再看数值差
#
# 每一格以 `(table, panel, row, metric)` 为键。缺数格的计算值保持 NaN，不是 0；论文 `<0.0001` 的 p 值参照只是上界，差值不解释为精确误差。1—10、A1—A5 每张表都出现在覆盖表中。
#
# Table 1、A2、A3、Fig 1/2/A1 的额外数据接口在下一份 notebook 中；这里的缺项状态会在那里补充。
#
computed_targets = pd.DataFrame(computed_rows)
assert not computed_targets.duplicated(['table','panel','row','metric']).any()
comparison_list = []
for table,target in paper_targets.items():
    reference = target.assign(table=table)
    compared = reference.merge(computed_targets,on=['table','panel','row','metric'],how='left',validate='one_to_one')
    compared['difference'] = compared['computed_value']-compared['paper_value']
    compared['status'] = compared['status'].fillna('尚缺同口径数据/扩展接口；未计算此格')
    comparison_list.append(compared)
comparison = pd.concat(comparison_list,ignore_index=True)
coverage = comparison.groupby('table').agg(paper_cells=('paper_value','count'),computed_cells=('computed_value','count'),min_months=('months','min'),max_months=('months','max'))
display(coverage)
for table in ['table2','table3','table4','table5','table6','table7','table8','table9','table10','a1','a4','a5']:
    print('原值 / 计算值 / 差值：',table)
    display(HTML(comparison[comparison['table'].eq(table)][['panel','row','metric','paper_value','computed_value','difference','months']].to_html(index=False,max_rows=None,max_cols=None)))


In [ ]:
# ## 结论检查：计算结果说什么，就记录什么
#
# “未拒绝零 alpha”不等于模型真实有效。为避免原文 5%/10% 混用，两个阈值并列；联合 GRS 与单项 t 各自报告。与原文不同的符号或显著性留下来追踪来源，不修改断点和窗口去拟合原表。
#
conclusion_rows = []
for panel in ['unconditional','size_neutral']:
    for model in ['capm','ff3','ch3','ch4']:
        estimates = anomaly_regressions[(panel,model)]
        estimates = estimates.loc[[name for name in main_anomalies if name in estimates.index]]
        conclusion_rows.append({'panel':panel,'model':model,'anomalies':len(estimates),'reject_5pct':int(estimates['significant_5pct'].sum()),'reject_10pct':int(estimates['significant_10pct'].sum()),'mean_abs_alpha_pct':estimates['alpha'].abs().mean(),'min_months':estimates['months'].min(),'max_months':estimates['months'].max()})
conclusions = pd.DataFrame(conclusion_rows)
display(conclusions)
print('以上是 RESSET 代理与缩短财务样本的实际结果，不能自动写成原文“8/10或10/10”。')
